# Lab 02 — Working with Data
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Week 2 lab · about 75 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/labs/lab02-working-with-data/Lab02_Working_with_Data.ipynb)

Machine learning starts with data. Before anyone trains a model, they *look* at the data: how much is there, what are the units, what looks normal, and what looks odd. Today you will explore a real laboratory dataset — **1,030 concrete mixes**, each crushed in a testing machine to measure its compressive strength. You will use **pandas** (tables) and **matplotlib** (plots), the two tools you will use in almost every lab from now on. Questions marked 🏠 may be finished after lab; they are still part of the lab.

### What you will be able to do
1. Load a CSV file into a pandas DataFrame and describe its size, columns, and summary statistics.
2. Select columns, filter rows, sort, and group a table to answer engineering questions.
3. Create a new column (the water–cement ratio) from existing columns.
4. Make histograms and scatter plots with a title, axis labels, and units, and say what they show.
5. Identify the features (X) and the target (y) for a prediction task, and spot unusual values.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `concrete_strength.csv` — 1,030 laboratory concrete tests (Yeh 1998, UCI Machine Learning Repository, CC BY 4.0). See the course `data/` folder README.

**AI-use label:** **AI-Assisted** — you may ask an AI tool for explanations or debugging help. Fill in the AI-use acknowledgment at the end. You are responsible for every line.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## Part 1 — Meet the data (10 min)
**pandas** is a Python library for tables. A pandas table is called a **DataFrame**: rows and columns, like a
spreadsheet, except you work with it in code. The setup cell imported it as `pd` (a short nickname everyone uses).

- `pd.read_csv(...)` reads a **CSV** file (comma-separated values — a plain-text table) into a DataFrame.
- `.head()` shows the first 5 rows. The bold numbers on the left (0, 1, 2, …) are the **row labels**.

In [ ]:
df = pd.read_csv(get_data("concrete_strength.csv"))
df.head()

Each row is **one concrete mix tested at one age**. The same recipe often appears several times, tested at
different ages (for example 3, 7, and 28 days).

| Column | Meaning | Units |
|---|---|---|
| `cement_kg_m3` | Portland cement | kg per m³ of concrete |
| `slag_kg_m3` | Blast-furnace slag (a steel-making by-product that behaves like cement) | kg/m³ |
| `fly_ash_kg_m3` | Fly ash (ash from coal power plants, also cement-like) | kg/m³ |
| `water_kg_m3` | Water | kg/m³ |
| `superplasticizer_kg_m3` | Superplasticizer (admixture that lets concrete flow with less water) | kg/m³ |
| `coarse_agg_kg_m3` | Coarse aggregate (gravel) | kg/m³ |
| `fine_agg_kg_m3` | Fine aggregate (sand) | kg/m³ |
| `age_days` | Age when the sample was tested | days |
| `strength_mpa` | Compressive strength — **what we want to predict** | MPa |

`.shape` gives the size of the table as **(rows, columns)**. It has no parentheses after it because it is a stored
property of the table, not an action.

In [ ]:
print(df.shape)
print("Rows (tests):", df.shape[0])
print("Columns:", df.shape[1])

`.describe()` computes summary statistics for every column at once:
- **count** — how many values; **mean** — the average; **std** (standard deviation) — the typical distance of values
  from the mean;
- **min / max** — smallest and largest; **25% / 50% / 75%** — percentiles: 50% (the **median**) means half the
  values are below it.

`.round(1)` just rounds the display to 1 decimal place.

In [ ]:
df.describe().round(1)

**✍️ Question:** From the table above: what are the mean, minimum, and maximum compressive strength (with units)? Commonly specified 28-day strengths for buildings are roughly 20–40 MPa (about 3,000–6,000 psi). Does the range in this dataset look plausible?

*Your answer:* 

## Part 2 — Select, filter, and sort (15 min)
### 2.1 Selecting columns
One column: write its name in quotes inside square brackets, `df["strength_mpa"]`. A single column is called a
**Series**. Series have useful methods such as `.mean()`, `.median()`, `.min()`, and `.max()`.

In [ ]:
strength = df["strength_mpa"]
print(f"Mean strength:   {strength.mean():.1f} MPa")
print(f"Median strength: {strength.median():.1f} MPa")
print(f"Strongest mix:   {strength.max():.1f} MPa")

To pick **several** columns, put a *list* of names inside the brackets — that is why you see double brackets `[[ ... ]]`.

In [ ]:
df[["cement_kg_m3", "water_kg_m3"]].head()

> 💡 **Hint:** Store the full table first: `mix_table = df[["cement_kg_m3", "water_kg_m3", "age_days", "strength_mpa"]]` (no `.head()` on this line). Then, on the last line, show it with `mix_table.head()`.

In [ ]:
# TODO: make a smaller table called mix_table with only 4 columns:
#       cement, water, age, and strength. Then show its first 5 rows.
mix_table = None   # replace None
mix_table

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert mix_table is not None, "mix_table is still None. Finish the TODO above first (replace None)."
assert set(mix_table.columns) == {"cement_kg_m3", "water_kg_m3", "age_days", "strength_mpa"},     f"Expected the 4 columns cement_kg_m3, water_kg_m3, age_days, strength_mpa. You have {list(mix_table.columns)}. Check the spelling."
assert len(mix_table) == 1030,     f"mix_table should keep all 1,030 rows, but it has {len(mix_table)}. Did you put .head() on the 'mix_table = ...' line? Store the full table; use .head() only to show it."
print("Correct!")

### 2.2 Filtering rows
A **condition** on a column gives `True` or `False` for *every* row:

In [ ]:
(df["age_days"] == 28).head()

Put the condition inside `df[ ... ]` to keep only the rows where it is `True`. (Note: `==` *compares*;
a single `=` *assigns*.)

Concrete strength is usually specified at **28 days** — the design strength f′c normally refers to 28-day tests — so
the 28-day group is the most important one.

In [ ]:
at28 = df[df["age_days"] == 28]
print(len(at28), "tests were done at 28 days")
at28.head(3)

Combine two conditions with `&` ("and") or `|` ("or"). Each condition needs its **own parentheses**. If you forget them, pandas stops with `TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]` — that message means: put each condition in its own ( ).

In [ ]:
young_rich = df[(df["age_days"] <= 7) & (df["cement_kg_m3"] > 400)]
print(len(young_rich), "tests at 7 days or younger with more than 400 kg/m³ of cement")

> 💡 **Hint:** Two conditions: `(df["age_days"] == 28)` and `(df["strength_mpa"] >= 40)`, joined with `&`, inside `df[ ... ]`.

In [ ]:
# TODO: keep only the tests done at 28 days that reached AT LEAST 40 MPa.
strong28 = None   # replace None with a filtered table

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert strong28 is not None, "strong28 is still None. Finish the TODO above first (replace None)."
assert len(strong28) == 146,     f"Expected 146 rows, but you have {len(strong28)}. 'At least 40' means >= (not >), and the age must be == 28."
print(f"Correct! 146 tests = {100 * 146 / 425:.0f}% of the 425 tests done at 28 days.")

### 2.3 Sorting
`df.sort_values("column")` sorts from smallest to largest. Add `ascending=False` for largest first.
Here are the five strongest tests:

In [ ]:
df.sort_values("strength_mpa", ascending=False).head(5)

> 💡 **Hint:** Same as the cell above, but leave out `ascending=False` (smallest first is the default).

In [ ]:
# TODO: show the 5 WEAKEST tests (lowest strength first).
weakest = None   # replace None
weakest

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert weakest is not None, "weakest is still None. Finish the TODO above first (replace None)."
assert len(weakest) == 5, f"weakest should have 5 rows, but it has {len(weakest)}. Add .head(5) (or .head()) at the end."
assert abs(weakest["strength_mpa"].iloc[0] - df["strength_mpa"].min()) < 1e-9,     "The first row should be the weakest test (2.33 MPa). Sort smallest first: leave out ascending=False."
print("Correct!")

**✍️ Question:** Compare the five strongest and the five weakest tests. What do the weakest have in common (look at `age_days`, `cement_kg_m3`, and `slag_kg_m3`)? What do the strongest have in common?

*Your answer:* 

## Part 3 — Groups and new columns (15 min)
### 3.1 Group by age
How does strength change with age? `groupby` splits the table into groups (one per age), then summarizes each
group. Read the line below as: *"group the rows by age; in each group take the strength column; report the count
and the mean."*

In [ ]:
by_age = df.groupby("age_days")["strength_mpa"].agg(["count", "mean"]).round(1)
by_age

The ages are now the row labels of `by_age` (called the **index**). Plot mean strength against age:

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(by_age.index, by_age["mean"], "o-")
plt.xlabel("Age at testing (days)")
plt.ylabel("Mean compressive strength (MPa)")
plt.title("Mean strength of each age group")
plt.show()

The jump between **90 days** and **91 days** looks strange. Is it real aging? Let's see whether the groups contain similar mixes.

> 💡 **Hint:** Copy the `groupby` line above, change the column to `"cement_kg_m3"`, and use `.mean()` instead of `.agg([...])`.

In [ ]:
# TODO: use groupby to find the MEAN CEMENT content (kg/m³) of each age group. Round to 1 decimal.
cement_by_age = None   # replace None
cement_by_age

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert cement_by_age is not None, "cement_by_age is still None. Finish the TODO above first (replace None)."
assert abs(cement_by_age[91] - 392.3) < 0.5 and abs(cement_by_age[90] - 284.1) < 0.5,     "Expected about 392.3 kg/m³ at 91 days and 284.1 kg/m³ at 90 days. Group by 'age_days' and take the 'cement_kg_m3' column."
print("Correct!")

**✍️ Question:** 🏠 **Finish at home** (still part of the lab, due Sunday). Describe how mean strength changes from 3 to 28 days. Then explain the jump from 90 days (40.5 MPa) to 91 days (69.8 MPa): does concrete really gain 29 MPa in one day? Use the counts and the cement amounts.

*Your answer:* 

### 3.2 A new column: the water–cement ratio
You can compute a new column from existing ones, just like math on numpy arrays in Lab 01: pandas does the
calculation row by row. To store it, assign to a new column name in square brackets:
`df["new_name"] = ...`

> 💡 **Hint:** Divide one column by the other: `df["water_kg_m3"] / df["cement_kg_m3"]`.

In [ ]:
# TODO: add a column "wc_ratio" = water divided by cement, for every row.
df["wc_ratio"] = None   # replace None
df[["water_kg_m3", "cement_kg_m3", "wc_ratio"]].head()

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert "wc_ratio" in df.columns and df["wc_ratio"].notna().all(),     "The wc_ratio column is missing or still empty. Finish the TODO above first (replace None with a formula)."
assert abs(df["wc_ratio"].mean() - 0.748) < 0.001,     f"The mean w/c should be about 0.748, but yours is {df['wc_ratio'].mean():.3f}. Did you divide water by cement (not cement by water)?"
print("Correct!")

> 🏗️ **Engineering note.** This ratio uses *cement only*. Many mixes here also contain slag and fly ash,
which act like cement. Engineers then use the **water-to-cementitious-materials ratio**,
w/cm = water ÷ (cement + slag + fly ash). Keep this in mind when you see w/c values above 1 below.

In [ ]:
df["wc_ratio"].describe().round(2)

> 💡 **Hint:** Filter first, then select the strength column, then take the mean: `df[df["wc_ratio"] < 0.5]["strength_mpa"].mean()`.

In [ ]:
# TODO: compute the mean strength (MPa) of mixes with wc_ratio BELOW 0.5,
#       and of mixes with wc_ratio ABOVE 1.0.
low_wc_mean = None    # MPa
high_wc_mean = None   # MPa
print(f"w/c < 0.5: {low_wc_mean} MPa   |   w/c > 1.0: {high_wc_mean} MPa")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert low_wc_mean is not None and high_wc_mean is not None, "A value is still None. Finish the TODO above first."
assert abs(low_wc_mean - 49.9) < 0.1,     f"Expected about 49.9 MPa for w/c below 0.5, got {low_wc_mean:.1f}. Use < 0.5, then take the mean of the strength_mpa column."
assert abs(high_wc_mean - 25.2) < 0.1,     f"Expected about 25.2 MPa for w/c above 1.0, got {high_wc_mean:.1f}. Use > 1.0, then take the mean of the strength_mpa column."
print("Correct! Low-w/c mixes are about twice as strong on average.")

## Part 4 — Plots (20 min)
### 4.1 Histogram
A **histogram** splits the range of values into bins (intervals) and counts how many values fall in each bin. It
shows the **distribution**: which values are common and which are rare.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["strength_mpa"], bins=20, edgecolor="white")
plt.xlabel("Compressive strength (MPa)")
plt.ylabel("Number of tests")
plt.title("Distribution of concrete strength (1,030 tests)")
plt.show()

**✍️ Question:** Where are most strengths? Is the histogram symmetric, or does it have a longer tail on one side?

*Your answer:* 

### 4.2 Scatter plots
A **scatter plot** draws one dot per row: x = one column, y = another. It shows whether two quantities move
together. `alpha=0.4` makes the dots partly see-through, so places where many dots overlap look darker.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df["cement_kg_m3"], df["strength_mpa"], alpha=0.4, s=12)
plt.xlabel("Cement (kg/m³)")
plt.ylabel("Compressive strength (MPa)")
plt.title("Strength vs cement content")
plt.show()

> 💡 **Hint:** Copy the cell above. Change the x column to `df["age_days"]` and update the x label and title.

In [ ]:
# TODO: scatter plot of strength (y-axis) vs age (x-axis). Include a title and axis labels with units.

> 💡 **Hint:** The w/c ratio has no units (kg ÷ kg), so you can label the axis "Water–cement ratio (–)" or "(kg water per kg cement)".

In [ ]:
# TODO: scatter plot of strength (y-axis) vs water-cement ratio (x-axis), with title and labels.

One number that summarizes *how close to a straight line* two columns are is the **correlation
coefficient** r: from −1 (perfect downward line) through 0 (no straight-line pattern) to +1 (perfect upward line).

In [ ]:
df[["cement_kg_m3", "age_days", "wc_ratio", "strength_mpa"]].corr()["strength_mpa"].round(2)

**✍️ Question:** 🏠 **Finish at home** (still part of the lab, due Sunday). Compare your three scatter plots. (a) Which input shows the clearest trend with strength, and in which direction? (b) Which relationships look **curved** rather than straight? (c) Why is there so much vertical spread at any single cement value?

*Your answer:* 

## Part 5 — Features, target, and unusual values (10 min)
### 5.1 Features (X) and target (y)
In machine learning, the **target** (y) is what we want to predict, and the **features** (X) are the inputs used
to predict it. Capital X because it is a table (many columns); lowercase y because it is a single column.

For this dataset: target = compressive strength; features = the 7 ingredient amounts + age.
`wc_ratio` is an **engineered feature**: we built it from two other columns using engineering knowledge.

In [ ]:
feature_cols = ["cement_kg_m3", "slag_kg_m3", "fly_ash_kg_m3", "water_kg_m3",
                "superplasticizer_kg_m3", "coarse_agg_kg_m3", "fine_agg_kg_m3", "age_days"]
print(len(feature_cols), "feature columns")

> 💡 **Hint:** `feature_cols` is already a list of names, so `df[feature_cols]` selects all of them. y is one column: `df["strength_mpa"]`.

In [ ]:
# TODO: make X (a table with the feature columns) and y (the strength column). Print their shapes.
X = None
y = None

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert X is not None and y is not None, "X or y is still None. Finish the TODO above first."
assert X.shape == (1030, 8), f"X should be 1,030 rows × 8 columns, but it is {X.shape}. Use df[feature_cols]."
assert y.shape == (1030,), f"y should be one column (shape (1030,)), but it is {y.shape}. Use single brackets: df['strength_mpa']."
print("Correct!  X shape:", X.shape, "  y shape:", y.shape)

**✍️ Question:** Why would it be a mistake to put `strength_mpa` inside X?

*Your answer:* 

### 5.2 Spotting unusual values
Real data has surprises. Two quick checks:

**1. Duplicates** — rows that are identical in every column. `df.duplicated()` marks each repeat as `True`
(the first copy stays `False`), and `.sum()` counts the `True` values.

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
# keep=False shows ALL copies, so we can see the groups of repeated rows
df[df.duplicated(keep=False)].sort_values("strength_mpa").head(9)

**2. Extreme values** — the highest water–cement ratios. Are they typing errors?

In [ ]:
high_wc = df[df["wc_ratio"] > 1.5]
high_wc[["cement_kg_m3", "slag_kg_m3", "fly_ash_kg_m3", "water_kg_m3", "age_days", "wc_ratio", "strength_mpa"]]

**✍️ Question:** 🏠 **Finish at home** (still part of the lab, due Sunday). (a) The data has 25 exact duplicate rows. Give one possible reason, and say what you would do with them before training a model. (b) Are the mixes with w/c above 1.5 errors? Use the slag and fly-ash columns and the w/cm note from Part 3 to explain.

*Your answer:* 

---
## 🎟️ Exit ticket
Three short questions you answer **alone, without AI (AI-Free)**, during the last 10 minutes of lab — even if you have not finished every part above. Use your own words, 1–3 sentences each. (If you must leave early, answer them on your own before you submit.)

**1.** In your own words, what does `df[df["age_days"] == 28]` do?

*Your answer:* 

**2.** For predicting concrete strength, what are the features (X) and the target (y)? Give one example of an engineered feature.

*Your answer:* 

**3.** Name one unusual thing you found in this dataset and what you would do about it before training a model.

*Your answer:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
**Due:** Sunday 11:59 pm (Arizona time) of this week, on Brightspace. Late work: see the syllabus late-work table (two grace days per semester, then −10% per day, up to 3 days).

1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.